# Fase 1: Ingestão de Dados e Análise Exploratória (EDA)
**Objetivo:** Realizar a leitura do dataset de 1 milhão de funcionários, tratar inconsistências e analisar os padrões das variáveis em relação à taxa de Attrition.

In [0]:
!pip install pandas numpy matplotlib seaborn scipy scikit-learn statsmodels missingno -q

In [0]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, ttest_ind

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import statsmodels.api as sm
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

sns.set_theme(style="whitegrid", context="notebook")

In [0]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def clip_int(x, low, high):
    return np.clip(np.round(x), low, high).astype(int)


def generate_ibm_hr_synthetic(n=1470, seed=42):
    """
    Gera uma base sintética inspirada no IBM HR Analytics Employee Attrition & Performance.

    A variável Attrition é gerada por uma função logística com efeitos interpretáveis:
    - maior attrition: OverTime, baixa satisfação, maior distância, menor idade,
      menor renda, solteiro, viagem frequente, pouco tempo na empresa.
    - menor attrition: maior senioridade, maior renda, mais estabilidade,
      stock option level, boa satisfação e bom equilíbrio vida-trabalho.
    """
    rng = np.random.default_rng(seed)

    employee_number = np.arange(1, n + 1)

    age = clip_int(rng.normal(37, 9, n), 18, 60)
    gender = rng.choice(["Male", "Female"], size=n, p=[0.60, 0.40])

    marital_status = []
    for a in age:
        if a < 28:
            marital_status.append(rng.choice(["Single", "Married", "Divorced"], p=[0.58, 0.37, 0.05]))
        elif a < 42:
            marital_status.append(rng.choice(["Single", "Married", "Divorced"], p=[0.23, 0.65, 0.12]))
        else:
            marital_status.append(rng.choice(["Single", "Married", "Divorced"], p=[0.11, 0.69, 0.20]))
    marital_status = np.array(marital_status)

    department = rng.choice(
        ["Research & Development", "Sales", "Human Resources"],
        size=n,
        p=[0.65, 0.30, 0.05]
    )

    education = rng.choice([1, 2, 3, 4, 5], size=n, p=[0.12, 0.20, 0.39, 0.24, 0.05])

    education_field = []
    for d in department:
        if d == "Research & Development":
            education_field.append(rng.choice(
                ["Life Sciences", "Medical", "Technical Degree", "Other"],
                p=[0.38, 0.35, 0.20, 0.07]
            ))
        elif d == "Sales":
            education_field.append(rng.choice(
                ["Marketing", "Life Sciences", "Other", "Technical Degree"],
                p=[0.52, 0.20, 0.20, 0.08]
            ))
        else:
            education_field.append(rng.choice(
                ["Human Resources", "Other", "Life Sciences"],
                p=[0.65, 0.25, 0.10]
            ))
    education_field = np.array(education_field)

    job_role = []
    for d in department:
        if d == "Research & Development":
            job_role.append(rng.choice(
                ["Research Scientist", "Laboratory Technician", "Manufacturing Director",
                 "Healthcare Representative", "Research Director", "Manager"],
                p=[0.34, 0.32, 0.12, 0.12, 0.04, 0.06]
            ))
        elif d == "Sales":
            job_role.append(rng.choice(
                ["Sales Executive", "Sales Representative", "Manager"],
                p=[0.70, 0.22, 0.08]
            ))
        else:
            job_role.append(rng.choice(
                ["Human Resources", "Manager"],
                p=[0.82, 0.18]
            ))
    job_role = np.array(job_role)

    total_working_years = clip_int((age - 18) * rng.uniform(0.45, 0.95, n) + rng.normal(0, 2.2, n), 0, 40)
    job_level_raw = 1 + (total_working_years / 9) + rng.normal(0, 0.7, n)
    job_level = clip_int(job_level_raw, 1, 5)

    years_at_company = np.array([
        rng.integers(0, max(1, twy + 1)) if twy > 0 else 0
        for twy in total_working_years
    ])
    years_at_company = np.minimum(years_at_company, total_working_years)

    years_in_current_role = np.array([
        rng.integers(0, yac + 1) if yac > 0 else 0
        for yac in years_at_company
    ])

    years_with_curr_manager = np.array([
        rng.integers(0, yac + 1) if yac > 0 else 0
        for yac in years_at_company
    ])

    years_since_last_promotion = np.array([
        rng.integers(0, min(yac + 1, 16)) if yac > 0 else 0
        for yac in years_at_company
    ])

    num_companies_worked = clip_int(rng.poisson(lam=np.maximum(1.2, total_working_years / 8)), 0, 9)
    distance_from_home = clip_int(rng.gamma(shape=2.0, scale=4.8, size=n), 1, 29)

    business_travel = rng.choice(
        ["Non-Travel", "Travel_Rarely", "Travel_Frequently"],
        size=n,
        p=[0.10, 0.70, 0.20]
    )

    environment_satisfaction = rng.choice([1, 2, 3, 4], size=n, p=[0.18, 0.19, 0.31, 0.32])
    job_satisfaction = rng.choice([1, 2, 3, 4], size=n, p=[0.17, 0.20, 0.32, 0.31])
    relationship_satisfaction = rng.choice([1, 2, 3, 4], size=n, p=[0.16, 0.21, 0.32, 0.31])
    job_involvement = rng.choice([1, 2, 3, 4], size=n, p=[0.08, 0.25, 0.50, 0.17])
    work_life_balance = rng.choice([1, 2, 3, 4], size=n, p=[0.08, 0.24, 0.45, 0.23])

    performance_rating = rng.choice([3, 4], size=n, p=[0.85, 0.15])
    percent_salary_hike = clip_int(rng.normal(13 + 3 * (performance_rating == 4), 3, n), 11, 25)

    stock_option_level = []
    for m in marital_status:
        if m == "Single":
            stock_option_level.append(rng.choice([0, 1, 2, 3], p=[0.58, 0.28, 0.10, 0.04]))
        else:
            stock_option_level.append(rng.choice([0, 1, 2, 3], p=[0.34, 0.41, 0.18, 0.07]))
    stock_option_level = np.array(stock_option_level)

    over_time = []
    for wlb, ji in zip(work_life_balance, job_involvement):
        p_ot = 0.18 + 0.08 * (wlb <= 2) + 0.05 * (ji >= 3)
        over_time.append(rng.choice(["No", "Yes"], p=[1 - p_ot, p_ot]))
    over_time = np.array(over_time)

    training_times_last_year = clip_int(rng.poisson(2.3, n), 0, 6)

    base_income = (
        1300
        + job_level * 2400
        + total_working_years * 105
        + (performance_rating == 4) * 600
        + rng.normal(0, 850, n)
    )

    role_bonus_map = {
        "Manager": 3000,
        "Research Director": 3500,
        "Manufacturing Director": 1300,
        "Healthcare Representative": 950,
        "Sales Executive": 850,
        "Research Scientist": 450,
        "Laboratory Technician": -250,
        "Sales Representative": -450,
        "Human Resources": 100
    }
    role_bonus = np.array([role_bonus_map[r] for r in job_role])

    monthly_income = clip_int(base_income + role_bonus, 1000, 22000)
    monthly_rate = clip_int(monthly_income * rng.uniform(1.2, 1.9, n) + rng.normal(0, 900, n), 2000, 27000)
    daily_rate = clip_int(rng.normal(800, 400, n), 100, 1500)
    hourly_rate = clip_int(rng.normal(66, 20, n), 30, 100)

    employee_count = np.ones(n, dtype=int)
    standard_hours = np.repeat(80, n)
    over_18 = np.repeat("Y", n)

    logit = (
        -1.85
        + 0.82 * (over_time == "Yes")
        + 0.52 * (business_travel == "Travel_Frequently")
        + 0.20 * (business_travel == "Travel_Rarely")
        + 0.42 * (marital_status == "Single")
        - 0.20 * (marital_status == "Married")
        + 0.024 * distance_from_home
        - 0.035 * (age - 35)
        - 0.000065 * (monthly_income - 6000)
        - 0.23 * (environment_satisfaction - 2.5)
        - 0.27 * (job_satisfaction - 2.5)
        - 0.18 * (work_life_balance - 2.5)
        - 0.19 * (job_involvement - 2.5)
        - 0.07 * years_at_company
        + 0.12 * num_companies_worked
        - 0.18 * stock_option_level
        + 0.28 * (job_role == "Sales Representative")
        + 0.18 * (job_role == "Laboratory Technician")
        - 0.10 * training_times_last_year
    )

    attrition_probability = sigmoid(logit)
    attrition = rng.binomial(1, attrition_probability)

    df = pd.DataFrame({
        "Age": age,
        "Attrition": np.where(attrition == 1, "Yes", "No"),
        "BusinessTravel": business_travel,
        "DailyRate": daily_rate,
        "Department": department,
        "DistanceFromHome": distance_from_home,
        "Education": education,
        "EducationField": education_field,
        "EmployeeCount": employee_count,
        "EmployeeNumber": employee_number,
        "EnvironmentSatisfaction": environment_satisfaction,
        "Gender": gender,
        "HourlyRate": hourly_rate,
        "JobInvolvement": job_involvement,
        "JobLevel": job_level,
        "JobRole": job_role,
        "JobSatisfaction": job_satisfaction,
        "MaritalStatus": marital_status,
        "MonthlyIncome": monthly_income,
        "MonthlyRate": monthly_rate,
        "NumCompaniesWorked": num_companies_worked,
        "Over18": over_18,
        "OverTime": over_time,
        "PercentSalaryHike": percent_salary_hike,
        "PerformanceRating": performance_rating,
        "RelationshipSatisfaction": relationship_satisfaction,
        "StandardHours": standard_hours,
        "StockOptionLevel": stock_option_level,
        "TotalWorkingYears": total_working_years,
        "TrainingTimesLastYear": training_times_last_year,
        "WorkLifeBalance": work_life_balance,
        "YearsAtCompany": years_at_company,
        "YearsInCurrentRole": years_in_current_role,
        "YearsSinceLastPromotion": years_since_last_promotion,
        "YearsWithCurrManager": years_with_curr_manager,
        "AttritionProbability_hidden": attrition_probability
    })

    return df


df = generate_ibm_hr_synthetic(n=100000, seed=42)
df.head()

In [0]:
# Removendo colunas com o mesmo valor para todos e identificadores
colunas_para_remover = ['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber', 'AttritionProbability_hidden']
df_clean = df.drop(columns=colunas_para_remover)

# Convertendo a variável Target (Attrition) para binário numérico (1 = Yes, 0 = No)
# Isso facilita muito a plotagem de gráficos e os modelos de Machine Learning futuros
df_clean['Attrition_Target'] = df_clean['Attrition'].apply(lambda x: 1 if x == 'Yes' else 0)

print(f"Dimensões do dataset após limpeza: {df_clean.shape[0]} linhas e {df_clean.shape[1]} colunas.")

In [0]:
# Calculando a taxa de Attrition
attrition_rate = df_clean['Attrition'].value_counts(normalize=True) * 100
print("Taxa de Rotatividade (Attrition):\n", attrition_rate.round(2))

# Gráfico de barras simples e elegante
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df_clean, x='Attrition', palette=['#2ECC71', '#E74C3C'])
plt.title('Distribuição de Attrition (TechCorp Brasil)', fontsize=14, fontweight='bold')
plt.ylabel('Total de Funcionários')
plt.xlabel('Saiu da Empresa? (Attrition)')

# Adicionando os rótulos de dados no topo das barras
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', fontsize=11, color='black', xytext=(0, 5),
                textcoords='offset points')
plt.show()

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Impacto da Renda Mensal
sns.boxplot(data=df_clean, x='Attrition', y='MonthlyIncome', palette='coolwarm', ax=axes[0])
axes[0].set_title('Renda Mensal vs Attrition')

# 2. Impacto da Idade
sns.kdeplot(data=df_clean, x='Age', hue='Attrition', fill=True, palette='coolwarm', ax=axes[1])
axes[1].set_title('Distribuição de Idade por Attrition')

# 3. Impacto de Horas Extras (OverTime)
# Calculando a porcentagem de attrition por quem faz hora extra
ot_attrition = df_clean.groupby('OverTime')['Attrition_Target'].mean() * 100
sns.barplot(x=ot_attrition.index, y=ot_attrition.values, palette='coolwarm', ax=axes[2])
axes[2].set_title('Taxa de Attrition vs Horas Extras')
axes[2].set_ylabel('% de Attrition')

plt.tight_layout()
plt.show()

In [0]:
print('demonstrando para o paulo')
print('tentando novamente')

In [0]:
print('demonstrando para o Gabriel')
print('tentando novamente')

In [0]:
print('teste v2')

In [0]:
print('teste v3 - teste')

In [0]:
print('teste v4 - teste')